# P4 · Cổ điển & Foundation — Bước 5

**Input (đọc):** data/processed/data_long.parquet

**Output (bạn phải tạo, đúng format):** data/outputs/forecasts/{naive,snaive,croston,sba,tsb,chronos_bolt,ttm}.parquet

**Code mẫu chạy được:** mở `notebooks/M5_reference.ipynb` phần **§5 và §6** — copy các cell sang đây rồi mở rộng.

> Quy tắc vàng: không sửa `common/`; không shuffle; tôn trọng trục thời gian; ML dùng lag ≥ 28. Forecast lưu vào `data/outputs/forecasts/<model>.parquet` với cột: `series_id,date,d_int,quantile_level,value,model`.

## Cell setup dùng chung (chạy trước) — nạp tham số cố định + dữ liệu
Các cell dưới đây GIỐNG NHAU ở mọi notebook, đảm bảo cả nhóm dùng chung 1 cấu hình & dữ liệu.

In [ ]:
import sys, subprocess, importlib, time, warnings, os, json, math, random
# --- 0.2 Import lõi + dò package nào dùng được ------------------------------
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42); random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4); plt.rcParams["axes.grid"] = True

def can_import(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

AVAILABLE = {
    "pyarrow":      can_import("pyarrow"),
    "lightgbm":     can_import("lightgbm"),
    "statsforecast":can_import("statsforecast"),
    "chronos":      can_import("chronos"),
    "torch":        can_import("torch"),
    "ttm":          can_import("tsfm_public"),
}
try:
    import torch
    AVAILABLE["cuda"] = torch.cuda.is_available()
except Exception:
    AVAILABLE["cuda"] = False

print("Package/phần cứng dùng được:")
for k, v in AVAILABLE.items():
    print(f"  {'✓' if v else '✗'} {k}: {v}")
PARQUET_ENGINE = "pyarrow" if AVAILABLE["pyarrow"] else None

In [ ]:
# --- Nạp THAM SỐ CỐ ĐỊNH từ common/config.yaml (nguồn chân lý duy nhất) -------
import yaml
from pathlib import Path
_c = yaml.safe_load(open("common/config.yaml", encoding="utf-8"))
CONFIG = dict(
    SEED=_c["seed"], HORIZON=_c["horizon"],
    SAMPLE_PER_SEGMENT=_c.get("sampling", {}).get("cap", 1000),
    TRAIN_END=_c["split"]["train_end"], VAL_END=_c["split"]["val_end"],
    TEST_END=_c["split"]["test_end"],
    QUANTILES=_c["quantiles"], ADI_TH=_c["sbc"]["adi_threshold"],
    CV2_TH=_c["sbc"]["cv2_threshold"],
    CHRONOS_MODEL=_c["foundation"]["chronos_model"],
    CHRONOS_CONTEXT=_c["foundation"]["chronos_context"],
    TTM_MODEL=_c["foundation"]["ttm_model"],
    TTM_CONTEXT=_c["foundation"]["ttm_context"],
)
CONFIG["TEST_START"] = CONFIG["VAL_END"] + 1
Q = CONFIG["QUANTILES"]
RAW = Path("data/raw"); OUT = Path("data/outputs"); FC = OUT / "forecasts"
OUT.mkdir(parents=True, exist_ok=True); FC.mkdir(parents=True, exist_ok=True)
PARQUET_ENGINE = "pyarrow"
def save_parquet(df, path):
    path = Path(path); df.to_parquet(path, index=False); return path
print("CONFIG nạp từ common/config.yaml:", {k: CONFIG[k] for k in ["SEED","HORIZON","TRAIN_END","VAL_END","TEST_END"]})

In [ ]:
# --- Nạp DỮ LIỆU CHUNG: long (P1 đã sinh) + mẫu + nhãn + calendar/giá ---------
import numpy as np, pandas as pd
calendar = pd.read_csv(RAW / "calendar.csv")
prices   = pd.read_csv(RAW / "sell_prices.csv")

_long_path = Path("data/processed/data_long.parquet")
assert _long_path.exists(), ("Chưa có data_long. Hãy chạy notebooks/P1.ipynb trước "
                             "(hoặc lấy từ người P1) để sinh data/processed/data_long.parquet.")
long = pd.read_parquet(_long_path)
long["date"] = pd.to_datetime(long["date"])

sample_ids = pd.read_csv("common/sample_ids.csv")["series_id"].tolist()
seg_labels = pd.read_parquet("common/segment_labels.parquet")

# schema ngày (train/val/test) cho cửa sổ gốc
schema = dict(
    train_days=[f"d_{i}" for i in range(1, CONFIG["TRAIN_END"]+1)],
    val_days=[f"d_{i}" for i in range(CONFIG["TRAIN_END"]+1, CONFIG["VAL_END"]+1)],
    test_days=[f"d_{i}" for i in range(CONFIG["VAL_END"]+1, CONFIG["TEST_END"]+1)],
)
# insample (<= val_end) dùng cho model cổ điển & FM
insample = long[long.d_int <= CONFIG["VAL_END"]].sort_values(["series_id","d_int"])
print(f"long: {long.shape} | mẫu: {len(sample_ids)} chuỗi | nhãn: {seg_labels.shape}")

In [ ]:
# --- 2.5 Bảng actuals cho test (dùng cho đánh giá) --------------------------
# actual test: các ngày d_TEST_START..d_TEST_END của chuỗi mẫu
test_days = schema["test_days"]
actual_test = (long[long.d.isin(test_days)]
               [["series_id","date","d_int","sales","segment"]]
               .rename(columns={"sales":"actual"})
               .reset_index(drop=True))
# train history (long) để tính scale RMSSE/pinball
train_hist = long[long.d_int <= CONFIG["VAL_END"]][["series_id","d_int","sales"]].copy()
print("actual_test:", actual_test.shape, "| train_hist:", train_hist.shape)

In [ ]:
# --- 3.1 Tiện ích scale (RMSSE & pinball) + trọng số theo chuỗi -------------
def _scales(y):
    '''scale MSE & MAE cua naive-1 tren insample, tinh tu lan ban dau tien (kieu M5).'''
    nz = np.nonzero(y)[0]
    if len(nz) < 2:
        return np.nan, np.nan
    y = y[nz[0]:]
    d = np.diff(y)
    if len(d) == 0:
        return np.nan, np.nan
    return float(np.mean(d**2)), float(np.mean(np.abs(d)))

scale_rmsse, scale_mae = {}, {}
for sid, g in train_hist.sort_values("d_int").groupby("series_id"):
    s2, s1 = _scales(g["sales"].to_numpy(dtype=np.float64))
    scale_rmsse[sid], scale_mae[sid] = s2, s1

# trọng số WSPL = doanh số 28 ngày insample cuối (kiểu M5 uncertainty)
last28 = long[(long.d_int > CONFIG["VAL_END"]-28) & (long.d_int <= CONFIG["VAL_END"])].copy()
last28["dollar"] = last28["sales"] * last28["sell_price"].fillna(0)
weight = last28.groupby("series_id")["dollar"].sum()
weight = (weight / weight.sum()).to_dict()   # chuẩn hoá tổng = 1
print(f"Đã tính scale cho {len(scale_rmsse)} chuỗi. Ví dụ scale_mae: "
      f"{list(scale_mae.items())[0]}")

In [ ]:
# --- 3.2 Dispersion NB theo chuỗi (từ insample) -----------------------------
mv = train_hist.groupby("series_id")["sales"].agg(["mean","var"])
def _r(row):
    m, v = row["mean"], row["var"]
    if v > m and m > 0:
        return m*m / (v - m)      # r hữu hạn -> NB
    return np.inf                 # -> Poisson
disp_r = mv.apply(_r, axis=1).to_dict()

def point_to_quantiles(point_df, name):
    '''point_df: series_id,date,d_int,value(mean) -> long 9 quantile theo NB/Poisson.'''
    out = []
    pf = point_df.copy()
    pf["value"] = pf["value"].clip(lower=0).astype(float)
    pf["r"] = pf["series_id"].map(disp_r).fillna(np.inf)
    mu = pf["value"].to_numpy()
    r  = pf["r"].to_numpy()
    finite = np.isfinite(r) & (r > 0)
    for u in Q:
        qv = np.empty_like(mu)
        # NB: mean=mu, size=r  -> p = r/(r+mu)
        if finite.any():
            rr = r[finite]; mm = mu[finite]
            p = rr / (rr + np.where(mm > 0, mm, 1e-9))
            qv[finite] = stats.nbinom.ppf(u, rr, p)
        if (~finite).any():
            qv[~finite] = stats.poisson.ppf(u, mu[~finite])
        block = pf[["series_id","date","d_int"]].copy()
        block["quantile_level"] = u
        block["value"] = np.nan_to_num(qv, nan=0.0)
        out.append(block)
    res = pd.concat(out, ignore_index=True)
    res["model"] = name
    return res[["series_id","date","d_int","quantile_level","value","model"]]
print("Sẵn sàng point_to_quantiles(). Ví dụ disp_r:", list(disp_r.items())[0])

In [ ]:
# --- 3.3 evaluate(): RMSSE, SPL/WSPL, coverage theo từng chuỗi --------------
COV_PAIRS = [(0.25,0.75,0.50),(0.165,0.835,0.67),(0.025,0.975,0.95),(0.005,0.995,0.99)]

def evaluate(fdf, name):
    '''fdf: long forecast (series_id,d_int,quantile_level,value). Tra per-series metrics.'''
    piv = (fdf.pivot_table(index=["series_id","d_int"], columns="quantile_level", values="value")
              .reset_index())
    m = piv.merge(actual_test[["series_id","d_int","actual","segment"]],
                  on=["series_id","d_int"], how="inner")
    rows = []
    for sid, g in m.groupby("series_id"):
        a = g["actual"].to_numpy(dtype=float)
        p50 = g[0.5].to_numpy(dtype=float) if 0.5 in g else np.full_like(a, np.nan)
        sr, sm = scale_rmsse.get(sid, np.nan), scale_mae.get(sid, np.nan)
        rmsse = np.sqrt(np.mean((a-p50)**2)/sr) if (sr and sr > 0) else np.nan
        spls = []
        for u in Q:
            if u in g:
                q = g[u].to_numpy(dtype=float)
                pin = np.mean(np.maximum(u*(a-q), (u-1)*(a-q)))
                spls.append(pin/sm if (sm and sm > 0) else np.nan)
        spl = np.nanmean(spls) if spls else np.nan
        covs = {}
        for lo, hi, nom in COV_PAIRS:
            if lo in g and hi in g:
                inside = ((a >= g[lo].to_numpy()) & (a <= g[hi].to_numpy())).mean()
                covs[f"cov{int(nom*100)}"] = inside
        rows.append(dict(series_id=sid, segment=g["segment"].iloc[0], model=name,
                         RMSSE=rmsse, SPL=spl, weight=weight.get(sid, np.nan), **covs))
    return pd.DataFrame(rows)
print("evaluate() sẵn sàng.")

## Việc của P4a — Cổ điển (Naive/SNaive/Croston/SBA/TSB)

In [ ]:
# --- 5.1 Naive & SeasonalNaive tự cài (điểm) -> NB quantile -----------------
DINT = dict(zip(pd.to_datetime(calendar["date"]),
                calendar["d"].str.slice(2).astype(int)))
test_dates = [d for d, i in sorted(DINT.items(), key=lambda kv: kv[1])
              if CONFIG["TEST_START"] <= i <= CONFIG["TEST_END"]]
H = CONFIG["HORIZON"]

insample = long[long.d_int <= CONFIG["VAL_END"]].sort_values(["series_id","d_int"])
classical_points = {}

# Naive: lặp giá trị cuối
last_val = insample.groupby("series_id")["sales"].last()
# SeasonalNaive: 7 giá trị cuối lặp lại
def _snaive(s):
    v = s.to_numpy()[-7:]
    if len(v) < 7: v = np.pad(v, (7-len(v),0))
    return np.tile(v, int(np.ceil(H/7)))[:H]
snaive_map = insample.groupby("series_id")["sales"].apply(_snaive)

def _expand(vals_map, name, per_series_horizon=False):
    rows = []
    for sid in sample_ids:
        if per_series_horizon:
            vals = vals_map[sid]
        else:
            vals = np.full(H, float(vals_map.get(sid, 0.0)))
        for k, (dt, v) in enumerate(zip(test_dates, vals)):
            rows.append((sid, dt, DINT[dt], float(v)))
    df = pd.DataFrame(rows, columns=["series_id","date","d_int","value"])
    return df

classical_points["naive"]  = _expand(last_val.to_dict(), "naive")
classical_points["snaive"] = _expand(snaive_map.to_dict(), "snaive", per_series_horizon=True)
print("Naive & SeasonalNaive xong.")

In [ ]:
# --- 5.2 Croston / SBA / TSB qua statsforecast ------------------------------
sf_runtime = {}
if AVAILABLE["statsforecast"]:
    t0 = time.time()
    from statsforecast import StatsForecast
    from statsforecast.models import CrostonClassic, CrostonSBA, TSB
    sf_train = (insample[["series_id","date","sales"]]
                .rename(columns={"series_id":"unique_id","date":"ds","sales":"y"}))
    models = [CrostonClassic(), CrostonSBA(), TSB(alpha_d=0.2, alpha_p=0.2)]
    sf = StatsForecast(models=models, freq="D", n_jobs=1)
    fcst = sf.forecast(h=H, df=sf_train)
    colmap = {"CrostonClassic":"croston", "CrostonSBA":"sba", "TSB":"tsb"}
    for col, nm in colmap.items():
        if col in fcst.columns:
            pdf = (fcst[["unique_id","ds",col]]
                   .rename(columns={"unique_id":"series_id","ds":"date",col:"value"}))
            pdf["date"] = pd.to_datetime(pdf["date"])
            pdf["d_int"] = pdf["date"].map(DINT)
            classical_points[nm] = pdf[["series_id","date","d_int","value"]]
    sf_runtime["statsforecast(3 models)"] = round(time.time()-t0, 1)
    print(f"Croston/SBA/TSB xong trong {sf_runtime['statsforecast(3 models)']}s")
else:
    print("⚠ Bỏ Croston/SBA/TSB (chưa cài statsforecast).")

In [ ]:
# --- 5.3 Bung point -> 9 quantile (NB) & lưu -------------------------------
classical_fc = {}
for nm, pdf in classical_points.items():
    q = point_to_quantiles(pdf.dropna(subset=["d_int"]), nm)
    classical_fc[nm] = q
    save_parquet(q, FC / f"{nm}.parquet")
print("Đã lưu forecast cổ điển:", list(classical_fc.keys()))

## Việc của P4b — Foundation (Chronos-Bolt, TTM zero-shot)

In [ ]:
# --- 6.1 Chuẩn bị context (insample cuối) cho FM ---------------------------
ctx_ids, ctx_list = [], []
for sid, g in insample.groupby("series_id"):
    ctx_ids.append(sid)
    ctx_list.append(g["sales"].to_numpy(dtype=np.float32))
fm_runtime, fm_log, fm_fc = {}, {}, {}

def _fm_to_long(name, quant_arr, ids):
    '''quant_arr: (n_series, H, 9) -> long forecast chuan.'''
    rows = []
    for i, sid in enumerate(ids):
        for h, dt in enumerate(test_dates):
            for j, u in enumerate(Q):
                rows.append((sid, dt, DINT[dt], u, max(0.0, float(quant_arr[i, h, j]))))
    df = pd.DataFrame(rows, columns=["series_id","date","d_int","quantile_level","value"])
    df["model"] = name
    return df
print(f"Context sẵn sàng cho {len(ctx_ids)} chuỗi.")

In [ ]:
# --- 6.2 Chronos-Bolt (quantile trực tiếp) ---------------------------------
if AVAILABLE["chronos"] and AVAILABLE["torch"]:
    try:
        import torch
        from chronos import BaseChronosPipeline
        t0 = time.time()
        pipe = BaseChronosPipeline.from_pretrained(
            CONFIG["CHRONOS_MODEL"], device_map="cpu", torch_dtype=torch.float32)
        C = CONFIG["CHRONOS_CONTEXT"]
        quants = np.zeros((len(ctx_ids), H, len(Q)), dtype=np.float32)
        BATCH = 64
        for s in range(0, len(ctx_ids), BATCH):
            ctx = [torch.tensor(ctx_list[k][-C:]) for k in range(s, min(s+BATCH, len(ctx_ids)))]
            ql, _mean = pipe.predict_quantiles(
                inputs=ctx, prediction_length=H, quantile_levels=list(Q))
            quants[s:s+len(ctx)] = ql.numpy()
        fm_fc["chronos_bolt"] = _fm_to_long("chronos_bolt", quants, ctx_ids)
        save_parquet(fm_fc["chronos_bolt"], FC / "chronos_bolt.parquet")
        fm_runtime["chronos_bolt"] = round(time.time()-t0, 1)
        fm_log["chronos_bolt"] = f"OK ({CONFIG['CHRONOS_MODEL']}, ctx={C})"
        print(f"✓ Chronos-Bolt xong trong {fm_runtime['chronos_bolt']}s")
    except Exception as e:
        fm_log["chronos_bolt"] = f"FAIL: {repr(e)[:220]}"
        print("✗ Chronos-Bolt lỗi:", fm_log["chronos_bolt"])
else:
    fm_log["chronos_bolt"] = "SKIP: thiếu chronos/torch"
    print("⚠", fm_log["chronos_bolt"])

In [ ]:
# --- 6.3 TTM (point -> NB quantile) ----------------------------------------
if AVAILABLE["ttm"] and AVAILABLE["torch"]:
    try:
        import torch
        from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction
        t0 = time.time()
        model = TinyTimeMixerForPrediction.from_pretrained(
            CONFIG["TTM_MODEL"], num_input_channels=1)
        model.eval()
        CL = int(getattr(model.config, "context_length", CONFIG["TTM_CONTEXT"]))
        PL = int(getattr(model.config, "prediction_length", 96))
        rows = []
        BATCH = 64
        with torch.no_grad():
            for s in range(0, len(ctx_ids), BATCH):
                chunk = range(s, min(s+BATCH, len(ctx_ids)))
                arrs, mu, sd = [], [], []
                for k in chunk:
                    v = ctx_list[k][-CL:].astype(np.float32)
                    if len(v) < CL: v = np.pad(v, (CL-len(v), 0))
                    m, sig = v.mean(), v.std() + 1e-6
                    mu.append(m); sd.append(sig)
                    arrs.append((v - m) / sig)
                past = torch.tensor(np.stack(arrs))[..., None]   # (B, CL, 1)
                out = model(past_values=past)
                pred = out.prediction_outputs.numpy()[:, :, 0]   # (B, PL)
                for bi, k in enumerate(chunk):
                    yhat = pred[bi, :H] * sd[bi] + mu[bi]
                    for h, dt in enumerate(test_dates):
                        rows.append((ctx_ids[k], dt, DINT[dt], max(0.0, float(yhat[h]))))
        ttm_point = pd.DataFrame(rows, columns=["series_id","date","d_int","value"])
        fm_fc["ttm"] = point_to_quantiles(ttm_point, "ttm")
        save_parquet(fm_fc["ttm"], FC / "ttm.parquet")
        fm_runtime["ttm"] = round(time.time()-t0, 1)
        fm_log["ttm"] = f"OK ({CONFIG['TTM_MODEL']}, CL={CL}, PL={PL})"
        print(f"✓ TTM xong trong {fm_runtime['ttm']}s")
    except Exception as e:
        fm_log["ttm"] = f"FAIL: {repr(e)[:220]}"
        print("✗ TTM lỗi:", fm_log["ttm"])
else:
    fm_log["ttm"] = "SKIP: thiếu granite-tsfm(tsfm_public)/torch"
    print("⚠", fm_log["ttm"])

print("\\nFM runtime:", fm_runtime)
print("FM log:", json.dumps(fm_log, ensure_ascii=False, indent=2))